# Lab 06｜能否建立因果結論 Experimental Design

<a href="https://colab.research.google.com/github/johnnychao/statistics-in-context-bilingual/blob/main/labs/colab/lab-06-experimental-design.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

> Statistics in Context · Unit 1 · 原創合成資料 · 不評量 Python 語法


## Goal

情境：學校以自願參與者進行晨間規劃活動，並在每個年級內隨機分派 intervention 與 control。

- 辨認 **random assignment、control、replication、blocking**。
- 檢查區集內人數與介入前平衡。
- 分清楚因果結論與推廣範圍：random assignment 支持哪一個？random sampling 又支持哪一個？


## Setup

依序執行儲存格即可，不需要撰寫或背誦 Python。若想重新開始，請在 Colab 選擇 **Runtime → Restart session and run all**。

本 Lab 使用原創合成資料；所有代碼與數值均不對應真實學生。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# 集中設定：一般情況只需修改這一格的參數。
DATA_RELATIVE_PATH = "data/public/morning_readiness_pilot.csv"
REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/johnnychao/statistics-in-context-bilingual/main"

LOCAL_REPO_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/content/statistics-in-context-bilingual"),
]


def load_repo_csv(relative_path):
    # 先找本機 repo，再讀 GitHub raw；失敗時提供繁中修復訊息。
    relative_path = Path(relative_path)
    for candidate_root in LOCAL_REPO_ROOT_CANDIDATES:
        candidate = candidate_root / relative_path
        if candidate.is_file():
            return pd.read_csv(candidate), str(candidate.resolve())

    remote_url = f"{REPO_RAW_BASE_URL}/{relative_path.as_posix()}"
    try:
        return pd.read_csv(remote_url), remote_url
    except Exception as exc:
        raise RuntimeError(
            "無法載入資料。請確認網路連線，或從 GitHub repo 根目錄執行此 Notebook。"
            f" 嘗試的遠端網址：{remote_url}。"
            " 若 repo 尚未發布，請先將 data/public 的 CSV 上傳至 main branch。"
        ) from exc


data, data_source = load_repo_csv(DATA_RELATIVE_PATH)
print(f"已載入 {len(data)} 筆資料｜Loaded {len(data)} rows")
print(f"來源 Source: {data_source}")


## Steps

### 1. 驗證研究設計

每個年級有 30 位合成參與者，應在區集內各分 15 位至兩組。


In [ ]:
design_table = pd.crosstab(data["grade"], data["assignment_group"])
display(design_table)

print("Participants per group:")
display(data["assignment_group"].value_counts().rename("n").to_frame())
print("All blocked by grade:", data["blocked_by_grade"].eq("yes").all())


### 2. 比較介入前與介入後改變

`change_score = post_readiness − baseline_readiness`。正值表示準備度提高。


In [ ]:
group_summary = (
    data.groupby("assignment_group")
    .agg(
        n=("participant_id", "count"),
        baseline_mean=("baseline_readiness", "mean"),
        post_mean=("post_readiness", "mean"),
        mean_change=("change_score", "mean"),
        sd_change=("change_score", "std"),
    )
)
display(group_summary.round(2))

observed_change_difference = (
    group_summary.loc["intervention", "mean_change"]
    - group_summary.loc["control", "mean_change"]
)
print(f"Difference in mean change (intervention − control) = {observed_change_difference:.2f} points")


### 3. 視覺化處理效果

箱形圖用來比較 change score 的中心、變異與重疊；解釋時仍須提到隨機分派與研究對象。


In [ ]:
FIGURE_ALT = (
    "Side-by-side boxplots of readiness change for control and morning-planning "
    "groups in a synthetic grade-blocked randomized experiment."
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=data,
    x="assignment_group",
    y="change_score",
    order=["control", "intervention"],
    color="#A8DADC",
    ax=ax,
)
sns.stripplot(
    data=data,
    x="assignment_group",
    y="change_score",
    order=["control", "intervention"],
    color="#1D3557",
    alpha=0.45,
    jitter=0.23,
    size=3,
    ax=ax,
)
ax.axhline(0, color="#D62828", linestyle="--", linewidth=1)
ax.set_title("Readiness change by randomized assignment (synthetic pilot)")
ax.set_xlabel("Randomized assignment group（組別）")
ax.set_ylabel("Change in readiness score（points）")
plt.tight_layout()
plt.show()
print(f"Alt text: {FIGURE_ALT}")


### 4. 看見 random assignment 的平衡作用

下方只重新洗牌組別標籤，觀察 baseline mean difference 在隨機分派下如何變動。這是設計概念模擬，不是本單元的正式顯著性檢定。


In [ ]:
# ✏️ 修改任務：把 RANDOMIZATION_SEED 改成另一個整數，比較分布是否大致相同。
RANDOMIZATION_SEED = 613
SIMULATIONS = 500
rng = np.random.default_rng(RANDOMIZATION_SEED)
baseline_values = data["baseline_readiness"].to_numpy()
group_size = int((data["assignment_group"] == "intervention").sum())

random_baseline_differences = []
for _ in range(SIMULATIONS):
    shuffled = rng.permutation(baseline_values)
    random_baseline_differences.append(
        shuffled[:group_size].mean() - shuffled[group_size:].mean()
    )

observed_baseline_difference = (
    group_summary.loc["intervention", "baseline_mean"]
    - group_summary.loc["control", "baseline_mean"]
)

RANDOMIZATION_FIGURE_ALT = (
    "Histogram of baseline mean differences from 500 simulated random assignments, "
    "with a vertical line marking the observed intervention-minus-control difference."
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(random_baseline_differences, bins=22, color="#457B9D", edgecolor="white", ax=ax)
ax.axvline(observed_baseline_difference, color="#E63946", linewidth=2, label="Observed baseline difference")
ax.set_title("Baseline differences produced by random assignment")
ax.set_xlabel("Difference in baseline means（intervention − control, points）")
ax.set_ylabel("Number of simulated assignments（次數）")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Alt text: {RANDOMIZATION_FIGURE_ALT}")
print(f"Observed baseline difference = {observed_baseline_difference:.2f} points")


<details>
<summary><strong>AP English Response frame</strong></summary>

> Because students were randomly assigned within grade blocks, a difference in readiness change can be attributed to the morning-planning intervention, subject to the study conditions. Blocking by grade helps control grade-to-grade variation. However, the participants were volunteers rather than a random sample, so the result should not be generalized to all students without caution.

</details>

請指出 control、replication 與 blocking 分別出現在資料的哪個欄位或人數安排中。


## Checks

確認隨機分派人數、區集配置與 change score 定義。


In [ ]:
assert len(data) == 120
assert (design_table == 15).all().all()
assert data["participant_id"].is_unique
assert np.allclose(data["post_readiness"] - data["baseline_readiness"], data["change_score"])
assert set(data["assignment_group"]) == {"control", "intervention"}
print("✅ Checks passed：每個年級區集 15/15 分派，change score 計算一致。")


## Next Steps

整合成果：完成雙語校務資料建議書，依序回答「我們觀察到什麼？可以推廣給誰？可以作因果結論嗎？下一步應蒐集什麼資料？」
